# Access GCS Bucket

In [2]:
# Run this cell first

import os
from google.colab import userdata
import json

# --- 1. Securely get the key from Colab Secrets ---
key_string = userdata.get('GCP_KEY')
key_data = json.loads(key_string)

# --- 2. Authenticate gsutil ---
# Write the key to a temporary file that gsutil can read
with open('temp_key.json', 'w') as f:
    json.dump(key_data, f)

#os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = 'temp_key.json'
!gcloud auth activate-service-account --key-file=temp_key.json

# --- 3. Access bucket directories ---
BUCKET_NAME = "asvspoof-la-data-bucket"
!gsutil ls gs://{BUCKET_NAME}

Activated service account credentials for: [colab-storage-reader@voiceauth-479800.iam.gserviceaccount.com]
gs://asvspoof-la-data-bucket/ASVspoof2019_LA_cm_protocols/
gs://asvspoof-la-data-bucket/ASVspoof2019_LA_dev/
gs://asvspoof-la-data-bucket/ASVspoof2019_LA_eval/
gs://asvspoof-la-data-bucket/ASVspoof2019_LA_train/


# Datasets

### Training

In [3]:
# -- Training Dataset Labels --

import pandas as pd
import os

# --- 1. Define Your Bucket and File Paths ---
BUCKET_NAME = "asvspoof-la-data-bucket" # Or your actual bucket name
PROTOCOL_DIR = "ASVspoof2019_LA_cm_protocols"
TRAIN_PROTOCOL_FILE = "ASVspoof2019.LA.cm.train.trn.txt"
KEY_FILE_NAME = 'temp_key.json'

# Construct the full GCS path
# gsutil paths are gs://BUCKET_NAME/OBJECT_NAME
gcs_path = f"gs://{BUCKET_NAME}/{PROTOCOL_DIR}/{TRAIN_PROTOCOL_FILE}"
print(f"Reading labels from: {gcs_path}\n")

storage_options = {'token': KEY_FILE_NAME}

# --- 2. Read the Protocol File with Pandas ---
# [cite_start]Based on the README, the columns are space-separated[cite: 5].
# We'll give them names based on the README's 5-column format:
# [cite_start]SPEAKER_ID AUDIO_FILE_NAME - SYSTEM_ID KEY [cite: 5]
column_names = ['SPEAKER_ID', 'AUDIO_FILE_NAME', 'BLANK', 'SYSTEM_ID', 'KEY']

# Read the file directly from GCS
# We use sep=' ' to indicate space-separated values
train_df = pd.read_csv(
    gcs_path,
    sep=' ',
    header=None,
    names=column_names,
    storage_options=storage_options,
)

# --- 3. Clean Up and Process the DataFrame ---

# We only need the file name and the label (KEY)
train_df = train_df[['AUDIO_FILE_NAME', 'KEY']]

# Convert string labels ('bonafide', 'spoof') to numbers (0, 1)
# This is a critical step for training.
train_df['LABEL'] = train_df['KEY'].apply(
    lambda x: 0 if x == 'bonafide' else 1
)

# We can now drop the original 'KEY' column
train_df = train_df.drop(columns=['KEY'])

# --- 4. Inspect the Result ---
print("Successfully loaded and processed labels:")
print(train_df.info())
print("\nFirst 5 rows:")
print(train_df.head())
print(f"\nTotal bonafide (real) samples: {(train_df['LABEL'] == 0).sum()}")
print(f"Total spoof (fake) samples: {(train_df['LABEL'] == 1).sum()}")

!rm temp_key.json

Reading labels from: gs://asvspoof-la-data-bucket/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt

Successfully loaded and processed labels:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25380 entries, 0 to 25379
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   AUDIO_FILE_NAME  25380 non-null  object
 1   LABEL            25380 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 396.7+ KB
None

First 5 rows:
  AUDIO_FILE_NAME  LABEL
0    LA_T_1138215      0
1    LA_T_1271820      0
2    LA_T_1272637      0
3    LA_T_1276960      0
4    LA_T_1341447      0

Total bonafide (real) samples: 2580
Total spoof (fake) samples: 22800


In [4]:
# Uninstall any conflicting versions
!pip uninstall -y tensorflow tensorflow-io

# Install the compatible pair for your new Colab runtime
!pip install tensorflow==2.16.1
!pip install tensorflow-io==0.37.1

Found existing installation: tensorflow 2.16.1
Uninstalling tensorflow-2.16.1:
  Successfully uninstalled tensorflow-2.16.1
Found existing installation: tensorflow-io 0.37.1
Uninstalling tensorflow-io-0.37.1:
  Successfully uninstalled tensorflow-io-0.37.1
  Using cached tensorflow-2.16.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.3 kB)
  Using cached ml_dtypes-0.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
Using cached tensorflow-2.16.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (589.9 MB)
Using cached ml_dtypes-0.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.2 MB)
  Attempting uninstall: ml-dtypes
    Found existing installation: ml_dtypes 0.5.4
    Uninstalling ml_dtypes-0.5.4:
      Successfully uninstalled ml_dtypes-0.5.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency 

In [5]:
!pip install ml_dtypes --upgrade

  Using cached ml_dtypes-0.5.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.9 kB)
Using cached ml_dtypes-0.5.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (5.0 MB)
  Attempting uninstall: ml_dtypes
    Found existing installation: ml-dtypes 0.3.2
    Uninstalling ml-dtypes-0.3.2:
      Successfully uninstalled ml-dtypes-0.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.16.1 requires ml-dtypes~=0.3.1, but you have ml-dtypes 0.5.4 which is incompatible.
tensorflow-decision-forests 1.12.0 requires tensorflow==2.19.0, but you have tensorflow 2.16.1 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tensorflow-text 2.19.0 requires tensorflow<2.20,>=2.19.0, but you have tensorflow 2.16.1 which is incompatible.
tf-keras 2.19.0 requires tensorflow<2.20,>=2.19

In [11]:
# -- Training Dataset Spectrograms --

import tensorflow as tf
import tensorflow_io as tfio  # <--- CHANGE 1: ADD THIS IMPORT
import numpy as np

# --- 1. Define Constants ---
BUCKET_NAME = "asvspoof-la-data-bucket"
AUDIO_DIR = "ASVspoof2019_LA_train"
SAMPLE_RATE = 16000
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128

# --- 2. Create the list of file paths and labels ---
audio_files = train_df['AUDIO_FILE_NAME'].tolist()
labels = train_df['LABEL'].tolist()

# --- 3. Build the tf.data.Dataset ---
dataset = tf.data.Dataset.from_tensor_slices((audio_files, labels))

# --- 4. Define the Preprocessing Function ---
def load_and_preprocess(audio_file_name, label):
    # 4a. Construct the full GCS path
    gcs_path = f"gs://{BUCKET_NAME}/{AUDIO_DIR}/" + audio_file_name + ".flac"

    # 4b. Read the file from GCS
    audio_binary = tf.io.read_file(gcs_path)

    # 4c. Decode the .flac file
    # <--- CHANGE 2: USE 'tfio'
    waveform = tfio.audio.decode_flac(audio_binary, dtype=tf.int16)

    # Convert to float and normalize
    waveform = tf.cast(waveform, tf.float32) / 32768.0

    # Remove the extra channel dimension
    waveform = tf.squeeze(waveform, axis=-1)

    # 4d. Create the Spectrogram
    stft = tf.signal.stft(
        waveform,
        frame_length=N_FFT,
        frame_step=HOP_LENGTH,
        fft_length=N_FFT
    )
    spectrogram = tf.abs(stft)

    # 4e. Create the Mel Spectrogram
    mel_filterbank = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=N_MELS,
        num_spectrogram_bins=stft.shape[-1],
        sample_rate=SAMPLE_RATE,
        lower_edge_hertz=0.0,
        upper_edge_hertz=(SAMPLE_RATE / 2.0)
    )
    mel_spectrogram = tf.tensordot(spectrogram, mel_filterbank, 1)

    # 4f. Convert to Log-Scale (dB)
    log_mel_spectrogram = tf.math.log(mel_spectrogram + 1e-6)

    # 4g. Return the processed "image" and its label
    return log_mel_spectrogram, label

# --- 5. Build the Final Pipeline ---
BATCH_SIZE = 32

train_dataset = (
    dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .shuffle(buffer_size=len(train_df))
    .padded_batch(BATCH_SIZE, padded_shapes=([None, N_MELS], []))
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

print("✅ tf.data pipeline built successfully.")
print(train_dataset)

✅ tf.data pipeline built successfully.
<_PrefetchDataset element_spec=(TensorSpec(shape=(None, None, 128), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>


### Validation

In [7]:
# -- Validation Dataset Labels --

import pandas as pd
import os

# --- 1. Define Your Bucket and File Paths ---
BUCKET_NAME = "asvspoof-la-data-bucket"
PROTOCOL_DIR = "ASVspoof2019_LA_cm_protocols"
# This time, we use the 'dev' protocol file
DEV_PROTOCOL_FILE = "ASVspoof2019.LA.cm.dev.trl.txt"
KEY_FILE_NAME = 'temp_key.json' # We'll re-use the key

# --- Create the key file (in case it was deleted) ---
# This is a good safety check
try:
    key_string = userdata.get('GCP_KEY')
    key_data = json.loads(key_string)
    with open(KEY_FILE_NAME, 'w') as f:
        json.dump(key_data, f)
except NameError:
    print("Assuming key file already exists...")


# Construct the full GCS path
gcs_path = f"gs://{BUCKET_NAME}/{PROTOCOL_DIR}/{DEV_PROTOCOL_FILE}"
print(f"Reading validation labels from: {gcs_path}\n")

# Explicitly pass the token to pandas
storage_options = {'token': KEY_FILE_NAME}

# --- 2. Read the Protocol File ---
# The README says the dev file has the same 5-column format
column_names = ['SPEAKER_ID', 'AUDIO_FILE_NAME', 'BLANK', 'SYSTEM_ID', 'KEY']

val_df = pd.read_csv(
    gcs_path,
    sep=' ',
    header=None,
    names=column_names,
    storage_options=storage_options
)

# --- 3. Clean Up and Process the DataFrame ---
val_df = val_df[['AUDIO_FILE_NAME', 'KEY']]
val_df['LABEL'] = val_df['KEY'].apply(
    lambda x: 0 if x == 'bonafide' else 1
)
val_df = val_df.drop(columns=['KEY'])

# --- 4. Inspect the Result ---
print("Successfully loaded and processed validation labels:")
print(val_df.info())
print("\nFirst 5 rows:")
print(val_df.head())

Reading validation labels from: gs://asvspoof-la-data-bucket/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt

Successfully loaded and processed validation labels:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24844 entries, 0 to 24843
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   AUDIO_FILE_NAME  24844 non-null  object
 1   LABEL            24844 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 388.3+ KB
None

First 5 rows:
  AUDIO_FILE_NAME  LABEL
0    LA_D_1047731      0
1    LA_D_1105538      0
2    LA_D_1125976      0
3    LA_D_1293230      0
4    LA_D_1340209      0


In [10]:
# -- Validation Dataset Spectrograms --

# --- 1. Define Constants for the Validation set ---
# We point to the DEV audio directory now
DEV_AUDIO_DIR = "ASVspoof2019_LA_dev"

# --- 2. Create the list of file paths and labels ---
# We use the 'val_df' DataFrame
val_audio_files = val_df['AUDIO_FILE_NAME'].tolist()
val_labels = val_df['LABEL'].tolist()

# --- 3. Build the tf.data.Dataset ---
val_dataset = tf.data.Dataset.from_tensor_slices((val_audio_files, val_labels))

# --- 4. Define the Preprocessing Function (slight modification) ---
# We create a new function just to point to the correct DEV folder.
# The actual processing logic is IDENTICAL.
def load_and_preprocess_validation(audio_file_name, label):
    # Point to the DEV folder
    gcs_path = f"gs://{BUCKET_NAME}/{DEV_AUDIO_DIR}/" + audio_file_name + ".flac"

    # The rest of this function is identical to the training one
    audio_binary = tf.io.read_file(gcs_path)
    waveform = tfio.audio.decode_flac(audio_binary, dtype=tf.int16)
    waveform = tf.cast(waveform, tf.float32) / 32768.0
    waveform = tf.squeeze(waveform, axis=-1)

    stft = tf.signal.stft(
        waveform,
        frame_length=N_FFT,
        frame_step=HOP_LENGTH,
        fft_length=N_FFT
    )
    spectrogram = tf.abs(stft)

    mel_filterbank = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=N_MELS,
        num_spectrogram_bins=stft.shape[-1],
        sample_rate=SAMPLE_RATE,
        lower_edge_hertz=0.0,
        upper_edge_hertz=(SAMPLE_RATE / 2.0)
    )
    mel_spectrogram = tf.tensordot(spectrogram, mel_filterbank, 1)
    log_mel_spectrogram = tf.math.log(mel_spectrogram + 1e-6)

    return log_mel_spectrogram, label

# --- 5. Build the Final Validation Pipeline ---
# Note: We .cache(), .padded_batch(), and .prefetch()
# We DO NOT .shuffle() the validation set.
val_dataset = (
    val_dataset.map(load_and_preprocess_validation, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .padded_batch(BATCH_SIZE, padded_shapes=([None, N_MELS], []))
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

print("\n✅ tf.data VALIDATION pipeline built successfully.")
print(val_dataset)

# --- 6. Clean up the key file ---
!rm {KEY_FILE_NAME}


✅ tf.data VALIDATION pipeline built successfully.
<_PrefetchDataset element_spec=(TensorSpec(shape=(None, None, 128), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>
rm: cannot remove 'temp_key.json': No such file or directory


# CNN Model

# RNN Model

In [13]:
!pip install keras

In [12]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Bidirectional, Masking, Dropout

# --- 1. Define Model Parameters ---
# Input shape: (Timesteps, Mel_bins)
# We use 'None' for timesteps because of padding
# We use 'N_MELS' (128) for the features
INPUT_SHAPE = (None, N_MELS)

# --- 2. Build the Model Architecture ---
model = Sequential([
    # This is the crucial layer we discussed.
    # It tells the model to ignore any timesteps that are all zeros.
    Masking(mask_value=0.0, input_shape=INPUT_SHAPE),

    # A Bidirectional GRU layer. 'return_sequences=True' means it outputs
    # the full sequence for the next layer to read.
    Bidirectional(GRU(64, return_sequences=True)),

    # A second Bidirectional GRU layer. This one (default) only
    # returns the *final* output after processing the whole sequence.
    Bidirectional(GRU(32)),

    # A standard "classifier" head
    Dense(16, activation='relu'),
    Dropout(0.3), # Dropout helps prevent overfitting

    # The final output layer.
    # '1' unit (for bonafide vs. spoof)
    # 'sigmoid' activation (squeezes the output between 0 and 1)
    Dense(1, activation='sigmoid')
])

# --- 3. Compile the Model ---
# This prepares the model for training
model.compile(
    # Adam is a great, all-purpose optimizer
    optimizer='adam',

    # This is the standard loss function for 0/1 classification
    loss='binary_crossentropy',

    # We want to track accuracy and AUC (Area Under the Curve),
    # which is a great metric for this type of problem.
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# --- 4. Print the Model Summary ---
# This shows you the layers, their shapes, and the number of parameters.
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking (Masking)               │ (None, None, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, None, 128)      │        74,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 64)             │        31,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 106,657 (416.63 KB)

 Trainable params: 106,657 (416.63 KB)

 Non-trainable params: 0 (0.00 B)